# Data Cleaning Essentials

## 1. Handling Missing Values

Missing values occur when data is not recorded or is unavailable. Handling them is crucial for accurate analysis and modeling.

### Key Points:
- **Mean Imputation**
  - Replace missing numerical values with the mean of the column.
  - Best for normally distributed data.
  - **Example:**  
    Column: `[10, 12, None, 14]` → Mean = `(10+12+14)/3 = 12` → Imputed: `[10, 12, 12, 14]`

- **Median Imputation**
  - Replace missing numerical values with the median of the column.
  - Robust to outliers and skewed data.
  - **Example:**  
    Column: `[10, 12, None, 20]` → Median = `12` → Imputed: `[10, 12, 12, 20]`

- **Mode Imputation**
  - Replace missing categorical values with the mode (most frequent value) of the column.
  - Useful for categorical data.
  - **Example:**  
    Column: `['M', 'F', None, 'F']` → Mode = `'F'` → Imputed: `['M', 'F', 'F', 'F']`

**Note:** Always analyze the data before choosing an imputation method.

---

## 2. Outlier Detection

Outliers are data points that are significantly different from the majority of the data. Detecting and handling them improves model accuracy.

### Key Points:
- **IQR (Interquartile Range) Method**
  - Calculate Q1 (25th percentile) and Q3 (75th percentile).
  - Outliers are values below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`.
- **Z-Score Method**
  - Standardize values and calculate Z-score.
  - Values with Z-score > 3 or < -3 are considered outliers.
- **Impact of Outliers**
  - Can distort mean and standard deviation.
  - May affect model performance, especially for regression algorithms.
- **Handling Outliers**
  - Remove, transform, or cap them based on analysis needs.

---

**Summary:**
- Missing value imputation and outlier detection are essential steps in data preprocessing.
- Correct handling improves data quality and model performance.


In [3]:
import csv

# Helper functions 
def mean(data):
    if not data:
        return None
    return sum(data) / len(data)

def median(data):
    if not data:
        return None
    data = sorted(data)
    n= len(data) 
    mid = n //2 
    if n%2 == 0:
        return (data[mid-1] + data[mid])/2
    else:
        return data[mid]
        
def mode(data):
    if not data:
        return None
    freq= {}
    for num in data:
        if num in freq:
            freq[num]+=1
        else:
            freq[num]=1
    mode1 = max(freq, key= freq.get)
    return mode1

def std(data):
    if not data:
        return None 
    m = mean(data)
    return (sum((x-m)**2 for x in data) /len(data)) ** 0.5

def get_quartile(data, q):
    if not data:
        return None
    if q not in (1, 2, 3):
        raise ValueError("q must be 1, 2, or 3")

    data = sorted(data)
    n = len(data)

    pos = q * (n+1) / 4

    if pos <= 1:
        return data[0]
    if pos >= n:
        return data[-1]

    lower = int(pos) - 1
    upper = lower + 1
    fraction = pos - int(pos)

    return data[lower] + fraction * (data[upper] - data[lower])


# Load Titanic dataset numerical columns (Age and Fare)
def load_titanic_numerical(file_path):
    numerical_data = {'Age': [], 'Fare': []}
    with open(file_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            age_val = row["Age"]
            fare_val =row["Fare"]
            numerical_data["Age"].append(float(age_val) if age_val != '' else None)
            numerical_data["Fare"].append(float(fare_val) if fare_val != '' else None)
        return numerical_data

'''
1. Create a fill_missing(data, strategy) function that fills missing values using:
• mean
• median
• mode
based on the selected strategy.
'''

def fill_missing(data, strategy="mean"):
    missing_count = data.count(None)
    if strategy == "mean":
        fill_val = mean([x for x in data if x is not None])
    elif strategy =="median":
        fill_val = median([x for x in data if x is not None])
    elif strategy =="mode":
        fill_val = mode(data)
    else:
        raise ValueError("Strategy must be in mean, medain or mode")
    filled_data = [x if x is not None else fill_val for x in data]
    print(f"Missing values handled using {strategy}: {missing_count}")
    return filled_data

'''
2. Create a drop_missing(data) function that removes records that contain missing values.
'''
def drop_missing(data):
    missing_count = data.count(None)
    cleaned_data =[x for x in data if x is not None]
    print(f"\nTotal values in data: {len(data)} ")
    print(f"Number of missing values removed: {missing_count}")
    print(f"Number of cleaned data: {len(cleaned_data)} ")
    return cleaned_data

'''
3. Implement a detect_outliers(data, method) function that identifies outliers using:
• IQR method
• Z-score method
'''
def detect_outliers(data,method="IQR"):
    outliers = []
    clean_data = [x for x in data if x is not None]
    if method == "IQR":
        Q3 = get_quartile(clean_data,3)
        Q1 = get_quartile(clean_data,1)
        IQR = Q3 - Q1 
        # print(Q1,Q3)
        lower_half = Q1 - 1.5* IQR
        upper_half = Q3 + 1.5 * IQR
        # print(lower_half,upper_half)
        for num in clean_data:
            if num < lower_half or num > upper_half:
                outliers.append(num)
    elif method =="Z-score":
        m = mean(clean_data)
        sd = std(clean_data)

        for x in clean_data:
            z = abs((x-m)/sd)
            if z > 3:
                outliers.append(x)
    else:
        raise ValueError("Method can only be Z-score and IQR")

    print(f"Number of outliers detected using {method}: {outliers}")
    print(f"Number of outliers detected using {method}: {len(outliers)}")
    return outliers

# Testing the functions with Titanic dataset
data_dict = load_titanic_numerical("Titanic-Dataset.csv")
age_data = data_dict['Age']
print("\nOriginal 'Age' Data (first 20 values):")
print(age_data[:20])

'''
Apply fill_missing() using all three strategies (mean, median, mode) on the same column
and compare how the filled values differ. Print a short comparison summary.
'''
# 1. Fill missing values using all strategies
age_filled_mean = fill_missing(age_data, 'mean')
age_filled_median = fill_missing(age_data, 'median')
age_filled_mode = fill_missing(age_data, 'mode')

# 2. Drop missing values
age_no_missing = drop_missing(age_data)

'''
After detecting outliers, create a new dataset where outliers are removed and compare the mean and median of the column before 
and after outlier removal.
'''
# 3. Detect outliers
outliers_iqr = detect_outliers(age_no_missing, 'IQR')
outliers_z = detect_outliers(age_no_missing, 'Z-score')

# 4. Comparison of filled values
print("\nComparison of filled values (first 20):")
for i in range(20):
    print(f"Original: {age_data[i]}, Mean: {age_filled_mean[i]}, Median: {age_filled_median[i]}, Mode: {age_filled_mode[i]}")


# Remove IQR outliers
age_cleaned_iqr = [x for x in age_no_missing if x not in outliers_iqr]

print("\n--- IQR Method ---")
print(f"Before Outlier Removal -> Mean: {mean(age_no_missing)}, Median: {median(age_no_missing)}")
print(f"After Outlier Removal  -> Mean: {mean(age_cleaned_iqr)}, Median: {median(age_cleaned_iqr)}")

# Remove Z-score outliers
age_cleaned_z = [x for x in age_no_missing if x not in outliers_z]

print("\n--- Z-score Method ---")
print(f"Before Outlier Removal -> Mean: {mean(age_no_missing)}, Median: {median(age_no_missing)}")
print(f"After Outlier Removal  -> Mean: {mean(age_cleaned_z)}, Median: {median(age_cleaned_z)}")


Original 'Age' Data (first 20 values):
[22.0, 38.0, 26.0, 35.0, 35.0, None, 54.0, 2.0, 27.0, 14.0, 4.0, 58.0, 20.0, 39.0, 14.0, 55.0, 2.0, None, 31.0, None]
Missing values handled using mean: 177
Missing values handled using median: 177
Missing values handled using mode: 177

Total values in data: 891 
Number of missing values removed: 177
Number of cleaned data: 714 
Number of outliers detected using IQR: [66.0, 71.0, 70.5, 71.0, 80.0, 70.0, 70.0, 74.0]
Number of outliers detected using IQR: 8
Number of outliers detected using Z-score: [80.0, 74.0]
Number of outliers detected using Z-score: 2

Comparison of filled values (first 20):
Original: 22.0, Mean: 22.0, Median: 22.0, Mode: 22.0
Original: 38.0, Mean: 38.0, Median: 38.0, Mode: 38.0
Original: 26.0, Mean: 26.0, Median: 26.0, Mode: 26.0
Original: 35.0, Mean: 35.0, Median: 35.0, Mode: 35.0
Original: 35.0, Mean: 35.0, Median: 35.0, Mode: 35.0
Original: None, Mean: 29.69911764705882, Median: 28.0, Mode: None
Original: 54.0, Mean: 54.0